In [246]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
navegador = webdriver.Chrome()

In [247]:
#Passo 1: abrir o navegador
#Passo 2: carregar a base
#Passo 3: para cada item da lista 
#Passo 4 : pesquisar para ver se existe
#Passo 5: se existir, pegar o preço e colocar na planilha

In [248]:
import pandas as pd
import openpyxl

In [249]:
df_produtos= pd.read_excel("Produtos.xlsx")
display(df_produtos)

,nome,autor,categoria,preco,link
0,Frankenstein,Mary Shelley,Classics,NaN,NaN
1,Romeo and Juliet,Shakespeare,Classics,NaN,NaN
2,The Great Gatsby,Fitzgerald,Classics,NaN,NaN
3,Algorithms to Live By,Brian Christian,Nonfiction,NaN,NaN
4,Sapiens,Yuval Harari,History,NaN,NaN
5,Smarter Faster Better,Charles Duhigg,Nonfiction,NaN,NaN


In [ ]:

def pesquisar_gutenberg(nome, autor, navegador):
    listas_palavras_autor = autor.lower().split(" ")
    navegador.get("https://gutenberg.org/")

    elemento = navegador.find_element("class name", "search-input")
    elemento.send_keys(nome)
    elemento.send_keys(Keys.ENTER)


    lista_resutados = navegador.find_elements("class name", "booklink")
    link = None
    preco = None

    for resultado in lista_resutados:
        texto = resultado.text.lower()

        if nome.lower() in texto:
            if any(palavra in texto for palavra in listas_palavras_autor):
                try:
                    link = resultado.find_element("class name", "link").get_attribute("href")
                except:
                    link = None
                preco = 0
                break

    return preco, link

def pesquisar_books_toscrape(nome, categoria, navegador):
    navegador.get("https://books.toscrape.com/")
    lista_categorias = navegador.find_elements("class name", "nav-list")

    try:
        lista_categorias[0].find_element("link text", categoria).click()
    except:
        print("Categoria não encontrada")

   
    link = None
    preco = None
    encontrado = False
    while not encontrado:
        lista_resultados = navegador.find_elements("class name", "product_pod")
        for resultado in lista_resultados:
            elemento_h3 =resultado.find_element("tag name", "h3")
            elemento_link = elemento_h3.find_element("tag name", "a")
            titulo = elemento_link.get_attribute("title")
            if nome.lower() in titulo.lower():
                encontrado = True
                link = elemento_link.get_attribute("href")
                preco = resultado.find_element("class name", "price_color").text
                break
        try:
            navegador.find_element("class name", "next").click()
        except:
            break
            
    return link, preco



for linha in df_produtos.index:
    nome = df_produtos.loc[linha, "nome"]
    autor = df_produtos.loc[linha, "autor"]
    categoria = df_produtos.loc[linha, "categoria"]

    link1, preco1 = pesquisar_gutenberg(nome, autor, navegador)
    print(nome,link1, preco1)
    if not link1:
        link2, preco2 = pesquisar_books_toscrape(nome, categoria, navegador)
        print(nome,link2, preco2)
        df_produtos.loc[linha, "preco"] = preco2
        df_produtos.loc[linha, "link"] = link2
    else:
        df_produtos.loc[linha, "preco"] = preco1
        df_produtos.loc[linha, "link"] = link1


Frankenstein 0 https://gutenberg.org/ebooks/84
Frankenstein None None
Romeo and Juliet 0 https://gutenberg.org/ebooks/1513
Romeo and Juliet None None
The Great Gatsby 0 https://gutenberg.org/ebooks/64317
The Great Gatsby None None
Algorithms to Live By None None
Algorithms to Live By https://books.toscrape.com/catalogue/algorithms-to-live-by-the-computer-science-of-human-decisions_880/index.html £30.81
Sapiens None None
Sapiens https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html £54.23
Smarter Faster Better None None
Smarter Faster Better None None
